1. Harte Strukturfilter.
2. Berechne positionsspezifische physikochemische Features, globale OT-Distanz und Strukturfeatures.
3. Trainiere Ridge, Gaussian Process und TabPFN mit LOOCV auf Aktivität und Tm.
4. Verwende die Modelle nicht allein, sondern als Exploit-Score.
5. Berechne zusätzlich Explore-Score über Distanz zu bekannten Varianten und pairwise diversity.
6. Wähle Kandidaten über Pareto-Front oder Cluster/MaxMin-Diversity aus.
7. Baue bewusst 2 bis 4 Kontrollvarianten ein.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path("/home/dwp46550/ba_nylon")

df = pd.read_csv(
    project_root / "matrices" / "NylC_Puetz_raw_data.CSV",
    sep=";",
    decimal=","
)
print(df.shape)

print(df.keys())

(36, 11)
Index(['variant_id', 'variant_class', 'mutations', 'n_mutations',
       'activity_pa6_1', 'activity_pa6_2', 'activity_pa6_3', 'activity_pa6_4',
       'tm_celsius_1', 'tm_celsius_2', 'strucuture_file'],
      dtype='object')


In [ ]:
import numpy as np
import pandas as pd
#calculates replicate mean 
activity_rep_cols = [
    col for col in df.columns
    if col.startswith("activity_pa6_")
]


df[activity_rep_cols] = df[activity_rep_cols].apply(
    pd.to_numeric,
    errors="coerce",
)


df["activity_n"] = df[activity_rep_cols].count(axis=1)

df["activity_pa6"] = df[activity_rep_cols].mean(
    axis=1,
    skipna=True,
)

df["activity_sd"] = df[activity_rep_cols].std(
    axis=1,
    skipna=True,
    ddof=1,
)

df["activity_sem"] = (
    df["activity_sd"]
    / np.sqrt(df["activity_n"])
)

print(df[
    ["variant_id", "activity_n", "activity_pa6",
     "activity_sd", "activity_sem"]
].head())

  variant_id  activity_n  activity_pa6  activity_sd  activity_sem
0         WT           3     77.666667     2.309401      1.333333
1       D99G           3    144.000000    19.313208     11.150486
2       D99V           3    147.666667     6.027714      3.480102
3       D99R           2    194.000000    14.142136     10.000000
4      F134W           2    217.000000     7.071068      5.000000


## define Activity Score


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler



AA_PROPERTIES_RAW = {
    # charge at physiological pH, Kyte-Doolittle hydrophobicity, side-chain volume Å³,
    # polarity, aromaticity, H-bond donor, H-bond acceptor, flexibility proxy
    "A": {"charge": 0,  "hydrophobicity": 1.8,  "volume": 88.6,  "polarity": 8.1,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.357},
    "R": {"charge": 1,  "hydrophobicity": -4.5, "volume": 173.4, "polarity": 10.5, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.529},
    "N": {"charge": 0,  "hydrophobicity": -3.5, "volume": 114.1, "polarity": 11.6, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.463},
    "D": {"charge": -1, "hydrophobicity": -3.5, "volume": 111.1, "polarity": 13.0, "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 1, "flexibility": 0.511},
    "C": {"charge": 0,  "hydrophobicity": 2.5,  "volume": 108.5, "polarity": 5.5,  "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 0, "flexibility": 0.346},
    "Q": {"charge": 0,  "hydrophobicity": -3.5, "volume": 143.8, "polarity": 10.5, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.493},
    "E": {"charge": -1, "hydrophobicity": -3.5, "volume": 138.4, "polarity": 12.3, "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 1, "flexibility": 0.497},
    "G": {"charge": 0,  "hydrophobicity": -0.4, "volume": 60.1,  "polarity": 9.0,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.544},
    "H": {"charge": 0.1,"hydrophobicity": -3.2, "volume": 153.2, "polarity": 10.4, "aromatic": 1, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.323},
    "I": {"charge": 0,  "hydrophobicity": 4.5,  "volume": 166.7, "polarity": 5.2,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.462},
    "L": {"charge": 0,  "hydrophobicity": 3.8,  "volume": 166.7, "polarity": 4.9,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.365},
    "K": {"charge": 1,  "hydrophobicity": -3.9, "volume": 168.6, "polarity": 11.3, "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 0, "flexibility": 0.466},
    "M": {"charge": 0,  "hydrophobicity": 1.9,  "volume": 162.9, "polarity": 5.7,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.295},
    "F": {"charge": 0,  "hydrophobicity": 2.8,  "volume": 189.9, "polarity": 5.2,  "aromatic": 1, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.314},
    "P": {"charge": 0,  "hydrophobicity": -1.6, "volume": 112.7, "polarity": 8.0,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.509},
    "S": {"charge": 0,  "hydrophobicity": -0.8, "volume": 89.0,  "polarity": 9.2,  "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.507},
    "T": {"charge": 0,  "hydrophobicity": -0.7, "volume": 116.1, "polarity": 8.6,  "aromatic": 0, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.444},
    "W": {"charge": 0,  "hydrophobicity": -0.9, "volume": 227.8, "polarity": 5.4,  "aromatic": 1, "hbond_donor": 1, "hbond_acceptor": 0, "flexibility": 0.305},
    "Y": {"charge": 0,  "hydrophobicity": -1.3, "volume": 193.6, "polarity": 6.2,  "aromatic": 1, "hbond_donor": 1, "hbond_acceptor": 1, "flexibility": 0.420},
    "V": {"charge": 0,  "hydrophobicity": 4.2,  "volume": 140.0, "polarity": 5.9,  "aromatic": 0, "hbond_donor": 0, "hbond_acceptor": 0, "flexibility": 0.386},
}


PROPERTY_COLUMNS = [
    "charge",
    "hydrophobicity",
    "volume",
    "polarity",
    "aromatic",
    "hbond_donor",
    "hbond_acceptor",
    "flexibility",
]


aa_property_df_raw = pd.DataFrame.from_dict(AA_PROPERTIES_RAW, orient="index")
aa_property_df_raw = aa_property_df_raw.loc[sorted(aa_property_df_raw.index), PROPERTY_COLUMNS]


scaler = StandardScaler()
aa_property_df_scaled = pd.DataFrame(
    scaler.fit_transform(aa_property_df_raw),
    index=aa_property_df_raw.index,
    columns=aa_property_df_raw.columns,
)


AA_PROPERTIES_SCALED = aa_property_df_scaled.to_dict(orient="index")


WT_POCKET = {
    99: "D",
    134: "F",
    304: "D",
    330: "R",
}


POSITIONS = [99, 134, 304, 330]






def aa_delta_features(wt_aa, mut_aa, position, scaled=True):
    if scaled:
        wt_props = AA_PROPERTIES_SCALED[wt_aa]
        mut_props = AA_PROPERTIES_SCALED[mut_aa]
    else:
        wt_props = AA_PROPERTIES_RAW[wt_aa]
        mut_props = AA_PROPERTIES_RAW[mut_aa]

    wt_props_raw = AA_PROPERTIES_RAW[wt_aa]
    mut_props_raw = AA_PROPERTIES_RAW[mut_aa]

    features = {}

    for col in PROPERTY_COLUMNS:
        features[f"pos{position}_delta_{col}"] = mut_props[col] - wt_props[col]

    features[f"pos{position}_charge_reversal"] = int(
        np.sign(wt_props_raw["charge"]) != np.sign(mut_props_raw["charge"])
        and wt_props_raw["charge"] != 0
        and mut_props_raw["charge"] != 0
    )

    features[f"pos{position}_aromatic_gain"] = int(
        mut_props_raw["aromatic"] > wt_props_raw["aromatic"]
    )

    features[f"pos{position}_aromatic_loss"] = int(
        mut_props_raw["aromatic"] < wt_props_raw["aromatic"]
    )

    return features


def build_pocket_physchem_features(mutant_pocket, wt_pocket=WT_POCKET):
    """
    mutant_pocket example:
    {99: "R", 134: "W", 304: "M", 330: "A"}

    Returns one feature dictionary for one variant.
    """

    features = {}

    for position in POSITIONS:
        wt_aa = wt_pocket[position]
        mut_aa = mutant_pocket.get(position, wt_aa)
        

        wt_aa = str(wt_aa).upper()
        mut_aa = str(mut_aa).upper()

        features[f"pos{position}_wt_aa"] = wt_aa
        features[f"pos{position}_mut_aa"] = mut_aa
        features[f"pos{position}_is_mutated"] = int(wt_aa != mut_aa)

        features.update(
            aa_delta_features(
                wt_aa=wt_aa,
                mut_aa=mut_aa,
                position=position,
                scaled=True,
            )
        )

    distance_cols = [f"pos{p}_distance_euclidean" for p in POSITIONS]

    features["n_mutations_pocket"] = sum(features[f"pos{p}_is_mutated"] for p in POSITIONS)#unnötig da wir in den BoltzGen varianten immer vier Mutationen haben 
    

    features["total_delta_charge"] = sum(features[f"pos{p}_delta_charge"] for p in POSITIONS) #passt
    features["total_delta_hydrophobicity"] = sum(features[f"pos{p}_delta_hydrophobicity"] for p in POSITIONS)#passt
    features["total_delta_volume"] = sum(features[f"pos{p}_delta_volume"] for p in POSITIONS)#passt
    features["total_delta_polarity"] = sum(features[f"pos{p}_delta_polarity"] for p in POSITIONS)#passt

    features["aromatic_gain_count"] = sum(features[f"pos{p}_aromatic_gain"] for p in POSITIONS)#passt
    features["aromatic_loss_count"] = sum(features[f"pos{p}_aromatic_loss"] for p in POSITIONS)#passt
    features["charge_reversal_count"] = sum(features[f"pos{p}_charge_reversal"] for p in POSITIONS) # das passt

    return features


def parse_mutation_string_to_pocket(mutation_string, wt_pocket=WT_POCKET):
    """
    Parses mutation strings like:
    'D99R/F134W/D304M/R330A'
    'D99R, F134W, D304M, R330A'
    'F134W D304M R330A'

    Returns a mutant pocket dictionary.
    """

    import re

    mutant_pocket = dict(wt_pocket)

    if pd.isna(mutation_string) or str(mutation_string).strip().lower() in ["", "wt", "wildtype", "wild type"]:
        return mutant_pocket

    pattern = r"([A-Z])(\d+)([A-Z])"
    matches = re.findall(pattern, str(mutation_string).upper())

    for wt_aa, pos, mut_aa in matches:
        pos = int(pos)

        if pos in mutant_pocket:
            expected_wt = wt_pocket[pos]

            if wt_aa != expected_wt:
                print(f"Warning: mutation {wt_aa}{pos}{mut_aa} does not match expected WT {expected_wt}{pos}")

            mutant_pocket[pos] = mut_aa

    return mutant_pocket


def add_physchem_features_from_mutation_column(df, mutation_col="mutations"):
    """
    Adds physicochemical mutation features to a dataframe.
    Requires a column with mutation strings.
    """

    feature_rows = []

    for _, row in df.iterrows():
        mutant_pocket = parse_mutation_string_to_pocket(row[mutation_col])
        feature_rows.append(build_pocket_physchem_features(mutant_pocket))

    feature_df = pd.DataFrame(feature_rows, index=df.index)

    return pd.concat([df.copy(), feature_df], axis=1)


def add_physchem_features_from_position_columns(
    df,
    pos_cols={
        99: "aa99",
        134: "aa134",
        304: "aa304",
        330: "aa330",
    },
):
    """
    Adds physicochemical mutation features to a dataframe.
    Requires one amino-acid column per position.
    Example columns: aa99, aa134, aa304, aa330
    """

    feature_rows = []

    for _, row in df.iterrows():
        mutant_pocket = {}

        for position, col in pos_cols.items():
            if col in df.columns and not pd.isna(row[col]):
                mutant_pocket[position] = str(row[col]).upper()
            else:
                mutant_pocket[position] = WT_POCKET[position]

        feature_rows.append(build_pocket_physchem_features(mutant_pocket))

    feature_df = pd.DataFrame(feature_rows, index=df.index)

    return pd.concat([df.copy(), feature_df], axis=1)


# Example usage with mutation strings:
# df_features = add_physchem_features_from_mutation_column(df, mutation_col="mutations")

# Example usage with position columns:
# df_features = add_physchem_features_from_position_columns(
#     df,
#     pos_cols={99: "aa99", 134: "aa134", 304: "aa304", 330: "aa330"}
# )

# Quick sanity check:
example_variant = {99: "R", 134: "W", 304: "M", 330: "A"}
example_features = build_pocket_physchem_features(example_variant)
pd.Series(example_features).head(30)



pos99_wt_aa                           D
pos99_mut_aa                          R
pos99_is_mutated                      1
pos99_delta_charge             4.466835
pos99_delta_hydrophobicity    -0.343484
pos99_delta_volume             1.544125
pos99_delta_polarity          -0.953333
pos99_delta_aromatic                0.0
pos99_delta_hbond_donor             2.0
pos99_delta_hbond_acceptor          0.0
pos99_delta_flexibility         0.22259
pos99_charge_reversal                 1
pos99_aromatic_gain                   0
pos99_aromatic_loss                   0
pos134_wt_aa                          F
pos134_mut_aa                         W
pos134_is_mutated                     1
pos134_delta_charge                 0.0
pos134_delta_hydrophobicity   -1.270892
pos134_delta_volume            0.939364
pos134_delta_polarity          0.076267
pos134_delta_aromatic               0.0
pos134_delta_hbond_donor            2.0
pos134_delta_hbond_acceptor         0.0
pos134_delta_flexibility      -0.111295


In [ ]:
import pandas as pd
import re
import ast

def mutations_to_pocket(mutations):
    pocket = WT_POCKET.copy()

    if mutations is None or (
        isinstance(mutations, float) and pd.isna(mutations)
    ):
        return pocket

    if isinstance(mutations, str):
        mutations = mutations.split(";")

    for mutation in mutations:
        mutation = mutation.strip()

        match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", mutation)

        if match is None:
            raise ValueError(f"mutational format not allowed: {mutation}")

        wt_aa, position, mutant_aa = match.groups()
        position = int(position)

        if position not in pocket:
            continue

        if pocket[position] != wt_aa:
            raise ValueError(
                f"{mutation}: wild type amino acid should be "
                f"{pocket[position]}."
            )

        pocket[position] = mutant_aa

    return pocket

In [ ]:
for variant, mutations in zip(df["variant_id"], df["mutations"]):
    pocket = mutations_to_pocket(mutations)

    print("variant:", variant)
    print("mutations:", mutations)
    print("pocket:", pocket)
    print("end")
    

variant: WT
mutations: nan
pocket: {99: 'D', 134: 'F', 304: 'D', 330: 'R'}
end
variant: D99G
mutations: D99G
pocket: {99: 'G', 134: 'F', 304: 'D', 330: 'R'}
end
variant: D99V
mutations: D99V
pocket: {99: 'V', 134: 'F', 304: 'D', 330: 'R'}
end
variant: D99R
mutations: D99R
pocket: {99: 'R', 134: 'F', 304: 'D', 330: 'R'}
end
variant: F134W
mutations: F134W
pocket: {99: 'D', 134: 'W', 304: 'D', 330: 'R'}
end
variant: F301L
mutations: F301L
pocket: {99: 'D', 134: 'F', 304: 'D', 330: 'R'}
end
variant: D304M
mutations: D304M
pocket: {99: 'D', 134: 'F', 304: 'M', 330: 'R'}
end
variant: D304E
mutations: D304E
pocket: {99: 'D', 134: 'F', 304: 'E', 330: 'R'}
end
variant: D304Q
mutations: D304Q
pocket: {99: 'D', 134: 'F', 304: 'Q', 330: 'R'}
end
variant: D304V
mutations: D304V
pocket: {99: 'D', 134: 'F', 304: 'V', 330: 'R'}
end
variant: D304W
mutations: D304W
pocket: {99: 'D', 134: 'F', 304: 'W', 330: 'R'}
end
variant: D304R
mutations: D304R
pocket: {99: 'D', 134: 'F', 304: 'R', 330: 'R'}
end
var

In [ ]:
df["pocket"] = df["mutations"].apply(
    parse_mutation_string_to_pocket
)

feature_df = pd.DataFrame(
    df["pocket"]
    .apply(build_pocket_physchem_features)
    .tolist(),
    index=df.index,
)

df_features = pd.concat(
    [df.copy(), feature_df],
    axis=1,
)

In [ ]:
feature_cols = [
    col for col in feature_df.columns
    if pd.api.types.is_numeric_dtype(feature_df[col])
]
#defines X and Y
X = df_features[feature_cols]
y = df_features["activity_pa6"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(feature_cols)

X shape: (36, 56)
y shape: (36,)
['pos99_is_mutated', 'pos99_delta_charge', 'pos99_delta_hydrophobicity', 'pos99_delta_volume', 'pos99_delta_polarity', 'pos99_delta_aromatic', 'pos99_delta_hbond_donor', 'pos99_delta_hbond_acceptor', 'pos99_delta_flexibility', 'pos99_charge_reversal', 'pos99_aromatic_gain', 'pos99_aromatic_loss', 'pos134_is_mutated', 'pos134_delta_charge', 'pos134_delta_hydrophobicity', 'pos134_delta_volume', 'pos134_delta_polarity', 'pos134_delta_aromatic', 'pos134_delta_hbond_donor', 'pos134_delta_hbond_acceptor', 'pos134_delta_flexibility', 'pos134_charge_reversal', 'pos134_aromatic_gain', 'pos134_aromatic_loss', 'pos304_is_mutated', 'pos304_delta_charge', 'pos304_delta_hydrophobicity', 'pos304_delta_volume', 'pos304_delta_polarity', 'pos304_delta_aromatic', 'pos304_delta_hbond_donor', 'pos304_delta_hbond_acceptor', 'pos304_delta_flexibility', 'pos304_charge_reversal', 'pos304_aromatic_gain', 'pos304_aromatic_loss', 'pos330_is_mutated', 'pos330_delta_charge', 'pos330

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    ElasticNet(max_iter=10000)
)

## testing feature stability

In [ ]:
import numpy as np
import pandas as pd

X = df_features[feature_cols].copy()

feature_quality = pd.DataFrame({
    "missing": X.isna().sum(),
    "infinite": np.isinf(X).sum(),
    "n_unique": X.nunique(),
    "variance": X.var(),
    "mean": X.mean(),
    "std": X.std(),
})

feature_quality["zero_variance"] = feature_quality["variance"] == 0
feature_quality["near_zero_variance"] = feature_quality["variance"] < 1e-10

feature_quality.sort_values(
    ["zero_variance", "n_unique"],
    ascending=[False, True],
)

,missing,infinite,n_unique,variance,mean,std,zero_variance,near_zero_variance
pos99_delta_aromatic,0,0,1,0.000000,0.000000,0.000000,True,True
pos99_aromatic_gain,0,0,1,0.000000,0.000000,0.000000,True,True
pos99_aromatic_loss,0,0,1,0.000000,0.000000,0.000000,True,True
pos134_delta_charge,0,0,1,0.000000,0.000000,0.000000,True,True
pos134_delta_aromatic,0,0,1,0.000000,0.000000,0.000000,True,True
pos134_delta_hbond_acceptor,0,0,1,0.000000,0.000000,0.000000,True,True
pos134_charge_reversal,0,0,1,0.000000,0.000000,0.000000,True,True
pos134_aromatic_gain,0,0,1,0.000000,0.000000,0.000000,True,True
pos134_aromatic_loss,0,0,1,0.000000,0.000000,0.000000,True,True
pos304_aromatic_loss,0,0,1,0.000000,0.000000,0.000000,True,True


In [ ]:
invalid_features = feature_quality.index[
    feature_quality["zero_variance"]
    | (feature_quality["missing"] > 0)
    | (feature_quality["infinite"] > 0)
].tolist()

print("problematic features:", invalid_features)

robust_feature_cols = [
    col for col in feature_cols
    if col not in invalid_features
]

problematic features: ['pos99_delta_aromatic', 'pos99_aromatic_gain', 'pos99_aromatic_loss', 'pos134_delta_charge', 'pos134_delta_aromatic', 'pos134_delta_hbond_acceptor', 'pos134_charge_reversal', 'pos134_aromatic_gain', 'pos134_aromatic_loss', 'pos304_aromatic_loss', 'pos330_delta_aromatic', 'pos330_aromatic_gain', 'pos330_aromatic_loss', 'aromatic_loss_count']


In [ ]:
from scipy.stats import spearmanr

def bootstrap_feature_stability(
    df,
    feature_cols,
    target_col="activity_pa6",
    n_bootstrap=2000,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    n = len(df)
    results = []

    for feature in feature_cols:
        observed_rho = spearmanr(
            df[feature],
            df[target_col],
            nan_policy="omit",
        ).statistic

        bootstrap_rhos = []

        for _ in range(n_bootstrap):
            indices = rng.integers(0, n, size=n)

            x_boot = df[feature].iloc[indices]
            y_boot = df[target_col].iloc[indices]

            if x_boot.nunique() < 2 or y_boot.nunique() < 2:
                continue

            rho = spearmanr(x_boot, y_boot).statistic

            if np.isfinite(rho):
                bootstrap_rhos.append(rho)

        bootstrap_rhos = np.asarray(bootstrap_rhos)

        if len(bootstrap_rhos) == 0:
            continue

        results.append({
            "feature": feature,
            "rho": observed_rho,
            "ci_low": np.percentile(bootstrap_rhos, 2.5),
            "ci_high": np.percentile(bootstrap_rhos, 97.5),
            "bootstrap_std": bootstrap_rhos.std(),
            "sign_stability": np.mean(
                np.sign(bootstrap_rhos) == np.sign(observed_rho)
            ),
        })

    return (
        pd.DataFrame(results)
        .sort_values(
            ["sign_stability", "bootstrap_std"],
            ascending=[False, True],
        )
    )

In [ ]:
stability_df = bootstrap_feature_stability(
    df_features,
    robust_feature_cols,
    target_col="activity_pa6",
)

stability_df

,feature,rho,ci_low,ci_high,bootstrap_std,sign_stability
13,pos134_delta_hbond_donor,0.827177,0.715526,0.867085,0.041368,1.000000
9,pos134_is_mutated,0.827177,0.711374,0.866918,0.041601,1.000000
12,pos134_delta_polarity,0.827177,0.715619,0.867309,0.041940,1.000000
11,pos134_delta_volume,0.827177,0.713663,0.866810,0.042037,1.000000
14,pos134_delta_flexibility,-0.827177,-0.867253,-0.714341,0.042572,1.000000
10,pos134_delta_hydrophobicity,-0.827177,-0.867314,-0.708148,0.043026,1.000000
40,aromatic_gain_count,-0.235949,-0.450999,-0.187397,0.071349,1.000000
20,pos304_delta_aromatic,-0.235949,-0.450754,-0.187349,0.071751,1.000000
35,n_mutations_pocket,0.780633,0.596202,0.887241,0.072084,1.000000
25,pos304_aromatic_gain,-0.235949,-0.450785,-0.187349,0.074112,1.000000


In [ ]:
support_rows = []

for feature in robust_feature_cols:
    counts = df_features[feature].value_counts(dropna=False)

    support_rows.append({
        "feature": feature,
        "n_unique": len(counts),
        "minimum_group_size": counts.min(),
        "maximum_group_size": counts.max(),
        "counts": counts.to_dict(),
    })

support_df = pd.DataFrame(support_rows)

stability_checked = stability_df.merge(
    support_df,
    on="feature",
    how="left",
)

stability_checked["sufficient_support"] = (
    stability_checked["minimum_group_size"] >= 4
)

stability_checked.sort_values(
    ["sufficient_support", "sign_stability"],
    ascending=[False, False],
)

,feature,rho,ci_low,ci_high,bootstrap_std,sign_stability,n_unique,minimum_group_size,maximum_group_size,counts,sufficient_support
0,pos134_delta_hbond_donor,0.827177,0.715526,0.867085,0.041368,1.000000,2,15,21,"{2.0: 21, 0.0: 15}",True
1,pos134_is_mutated,0.827177,0.711374,0.866918,0.041601,1.000000,2,15,21,"{1: 21, 0: 15}",True
2,pos134_delta_polarity,0.827177,0.715619,0.867309,0.041940,1.000000,2,15,21,"{0.07626660785666073: 21, 0.0: 15}",True
3,pos134_delta_volume,0.827177,0.713663,0.866810,0.042037,1.000000,2,15,21,"{0.9393635381458998: 21, 0.0: 15}",True
4,pos134_delta_flexibility,-0.827177,-0.867253,-0.714341,0.042572,1.000000,2,15,21,"{-0.11129480328715591: 21, 0.0: 15}",True
5,pos134_delta_hydrophobicity,-0.827177,-0.867314,-0.708148,0.043026,1.000000,2,15,21,"{-1.2708924859340271: 21, 0.0: 15}",True
11,pos330_delta_volume,-0.478756,-0.726274,-0.165313,0.145134,0.998000,3,5,25,"{0.0: 25, -2.1017949349544134: 6, -0.733645401...",True
13,pos330_delta_hydrophobicity,0.478756,0.135758,0.725623,0.151782,0.996500,3,5,25,"{0.0: 25, 2.163952070644425: 6, 0.343484455657...",True
14,pos330_delta_flexibility,-0.478756,-0.729325,-0.151059,0.150107,0.996000,3,5,25,"{0.0: 25, -2.12696735171009: 6, -0.44517921314...",True
15,pos330_charge_reversal,0.449903,0.128771,0.688500,0.146572,0.994500,2,11,25,"{0: 25, 1: 11}",True


In [ ]:
constant_features = feature_quality.index[
    feature_quality["n_unique"] == 1
].tolist()

X_reduced = X.drop(columns=constant_features)

print(constant_features)

['pos99_delta_aromatic', 'pos99_aromatic_gain', 'pos99_aromatic_loss', 'pos134_delta_charge', 'pos134_delta_aromatic', 'pos134_delta_hbond_acceptor', 'pos134_charge_reversal', 'pos134_aromatic_gain', 'pos134_aromatic_loss', 'pos304_aromatic_loss', 'pos330_delta_aromatic', 'pos330_aromatic_gain', 'pos330_aromatic_loss', 'aromatic_loss_count']


In [ ]:
position_support = []

for position in POSITIONS:
    aa_series = df_features["pocket"].apply(
        lambda pocket: pocket[position]
    )

    counts = aa_series.value_counts()

    for aa, count in counts.items():
        activity = df_features.loc[
            aa_series == aa,
            "activity_pa6",
        ]

        position_support.append({
            "position": position,
            "amino_acid": aa,
            "count": count,
            "mean_activity": activity.mean(),
            "median_activity": activity.median(),
            "std_activity": activity.std(),
        })

position_support = pd.DataFrame(position_support)

position_support.sort_values(
    ["position", "count"],
    ascending=[True, False],
)

,position,amino_acid,count,mean_activity,median_activity,std_activity
0,99,D,21,224.305556,202.333333,115.837666
1,99,R,7,254.428571,248.000000,83.516180
2,99,G,4,269.500000,262.000000,108.987767
3,99,V,4,236.166667,203.000000,106.479523
4,134,W,21,301.039683,258.000000,90.105061
5,134,F,15,146.150000,140.666667,35.149570
6,304,M,14,300.297619,262.000000,104.816612
7,304,D,10,183.550000,147.333333,91.721736
8,304,E,2,205.458333,205.458333,48.377889
9,304,V,2,166.916667,166.916667,50.086730


In [ ]:
exclude_features = constant_features + [
    "pos304_delta_aromatic",
    "pos304_aromatic_gain",
    "aromatic_gain_count",
    "total_delta_charge",
    "total_delta_hydrophobicity",
    "total_delta_volume",
    "total_delta_polarity",
]

## predict activity

In [ ]:
exclude_prefixes = (
    "total_",
    "aromatic_gain_count",
    "aromatic_loss_count",
    "charge_reversal_count",
    "n_mutations_pocket",
)

model_features = [
    col for col in feature_cols
    if col not in constant_features
    and not col.startswith(exclude_prefixes)
]

X = df_features[model_features].copy()
y = df_features["activity_pa6"].copy()

print(X.shape)

(36, 35)


In [18]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel,
)
from sklearn.model_selection import (
    LeaveOneOut,
    KFold,
    GridSearchCV,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from scipy.stats import pearsonr, spearmanr

import numpy as np
import pandas as pd

In [19]:
gp_kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e3))
    + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))
)

model_specs = {
    "Ridge": {
        "estimator": Ridge(),
        "params": {
            "model__alpha": np.logspace(-4, 4, 40),
        },
    },

    "ElasticNet": {
        "estimator": ElasticNet(max_iter=50000),
        "params": {
            "model__alpha": np.logspace(-3, 2, 25),
            "model__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9, 1.0],
        },
    },

    "RandomForest": {
        "estimator": RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
        ),
        "params": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [None, 3, 5],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", 0.7],
        },
    },

    "GaussianProcess": {
        "estimator": GaussianProcessRegressor(
            kernel=gp_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=42,
        ),
        "params": {},
    },
}

In [ ]:
outer_cv = LeaveOneOut()
all_predictions = {}
best_parameters = {}

for model_name, specification in model_specs.items():
    print("Evaluating:", model_name)

    predictions = np.zeros(len(y))
    fold_parameters = []

    for train_idx, test_idx in outer_cv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", clone(specification["estimator"])),
        ])

        if specification["params"]:
            inner_cv = KFold(
                n_splits=5,
                shuffle=True,
                random_state=42,
            )

            search = GridSearchCV(
                pipeline,
                specification["params"],
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                n_jobs=-1,
            )

            search.fit(X_train, y_train)
            fitted_model = search.best_estimator_
            fold_parameters.append(search.best_params_)

        else:
            fitted_model = pipeline.fit(X_train, y_train)

        predictions[test_idx] = fitted_model.predict(X_test)

    all_predictions[model_name] = predictions
    best_parameters[model_name] = fold_parameters

Evaluating: Ridge


In [ ]:
from tabpfn import TabPFNRegressor

tabpfn_predictions = np.zeros(len(y))

for train_idx, test_idx in outer_cv.split(X):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]

    tabpfn = TabPFNRegressor(device="auto")
    tabpfn.fit(X_train, y_train)

    tabpfn_predictions[test_idx] = tabpfn.predict(X_test)

all_predictions["TabPFN"] = tabpfn_predictions

In [ ]:
comparison_rows = []

for model_name, predictions in all_predictions.items():
    comparison_rows.append({
        "model": model_name,
        "R2": r2_score(y, predictions),
        "MAE": mean_absolute_error(y, predictions),
        "RMSE": mean_squared_error(y, predictions) ** 0.5,
        "Pearson": pearsonr(y, predictions).statistic,
        "Spearman": spearmanr(y, predictions).statistic,
    })

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("MAE")
    .reset_index(drop=True)
)

model_comparison

,model,R2,MAE,RMSE,Pearson,Spearman
0,RandomForest,0.568410,52.573662,68.365694,0.754992,0.780023
1,GaussianProcess,0.553015,52.729963,69.574312,0.744309,0.763934
2,Ridge,0.573843,55.888457,67.934006,0.764355,0.772429
3,ElasticNet,0.556354,55.978963,69.313955,0.755306,0.710259
4,TabPFN,0.550080,56.005367,69.802346,0.741980,0.746686


In [ ]:
prediction_df = pd.DataFrame({
    "variant_id": df_features["variant_id"],
    "observed": y,
    **all_predictions,
})

prediction_df

,variant_id,observed,Ridge,ElasticNet,RandomForest,GaussianProcess,TabPFN
0,WT,77,159.349491,157.532067,137.186034,134.680464,150.158966
1,D99G,144,143.656587,158.222601,138.467562,129.875453,146.990326
2,D99V,148,121.291476,153.300652,129.939914,138.063511,146.251404
3,D99R,194,132.527697,152.839475,134.215970,148.109107,140.670761
4,F134W,217,243.173156,254.713127,243.994296,236.392722,278.481384
5,F301L,118,155.015137,156.304465,109.375817,122.297828,141.008163
6,D304M,234,183.524914,156.774879,140.447500,166.367484,140.605423
7,D304E,171,159.143494,151.555363,126.700393,130.740031,142.180496
8,D304Q,157,156.700806,152.815871,144.253726,133.851037,143.048920
9,D304V,132,192.281391,173.952053,166.589167,178.515616,154.098480


In [ ]:
#generate Vector for Wildtype

# WT_pocket = [
#   vector(AA_WT_99),
#   vector(AA_WT_134),
#   vector(AA_WT_304),
#   vector(AA_WT_330)
# ]

#generate Vector for each lab tested variant interate over .csv file

# for ...
# Mut_pocket = [
#   vector(AA_mut_99),
#   vector(AA_mut_134),
#   vector(AA_mut_304),
#   vector(AA_mut_330)
# ]



In [ ]:
#calculate OT cost matrix

#same mass for each position

# WT_mass = [0.25, 0.25, 0.25, 0.25]
# Mut_mass = [0.25, 0.25, 0.25, 0.25]

#OT_distance_strucuture weighted = optimal_transport_structure(WT_mass, Mut_mass, cost_matrix)

In [ ]:
#OT_activity_informed= optimal_transport_structure(WT_mass, Mut_mass, cost_matrix)

# known_benefitial_positions_D99=[R,G,V]
# known_benefitial_positions_R330=[A,Q]
# known_benefitial_positions_D304=[M,E,Q,L,V,R,M]
# known_benefitial_positions_R134=[W]

In [ ]:
#simple models

#linear: activity ~ OT_distance
#polynom: activity ~ OT_distance + OT_distance²

#linear: Tm ~ OT_distance
#polynom: Tm ~ OT_distance + OT_distance²

#LOOC
# für jede Variante k:
#     trainiere Modell auf 35 Varianten
#     sage Aktivität/Tm der entfernten Variante k vorher
# am Ende:
#     berechne MAE, Spearman, Pearson, R²

In [ ]:
#more complex models

# Gaussian Process
# Random Forest
# TabPFN
# ElasticNet oder Ridge

## calculate Wasserstein difference between WT and mutant

In [ ]:
# spocket=gauß-score of Wasserstein difference

## calculate diversity score based on the wasserstein difference between variants

In [ ]:
#S_diversity(x)= min W(x,y)

## Calculate final ranking score for final ranking

In [ ]:
weights= [0.33, 0.33, 0.33] #can be adjusted: heuristic (could be learned?)

#score=weights[0]*S_structure+weights[1]*S_pocket+weights[2]*S_diversity